# UKFによる非線形システムの推定

- 状態推定
- 外乱推定
- パラメータ推定

## 概要

UKFは非線形システムの推定に用いられるアルゴリズムである。



### 解きたい問題

UKFは不確かな状態$x$を非線形な状態方程式$f$に通した後、通過した状態の平均と分散により、その状態の推定値を求める方法である。

例えば、1次元の状態$x_k$を考え、それが以下の平均$\mu_k$,分散$P_k$の正規分布に従う確率変数だとする。

$$
x_k \sim \mathcal{N}(\mu_k, P_k)
$$

この状態$x_k$を、非線形な状態方程式$f$を通過させると、次の状態$x_{k+1}$は以下のようになる。

$$
x_{k+1} = f(x_k)
$$

この状態$x_{k+1}$は確率変数である。この状態の推定値として$\mu_{k+1}$を設定する。 

$$
\mu_{k+1} = E[x_{k+1}] = E[f(x_k)]
$$

また、この推定がどれくらい不確かなのかを求めるために、分散$P_{k+1}$も求める。

$$
P_{k+1} = E[(f(x_k) - \mu_{k+1})^2]
$$

$f(x_k)$が以下のように線形である場合、以下のように平均と分散を計算できる。

$$
x_{k+1} = f(x_k) = ax_k + b
$$

- 平均

    $$
    \mu_{{k+1}} = E[f(x_k)] = E[a x_k + b] = aE[x_k] + b = a \mu_k + b
    $$

    これより、平均値を状態方程式へ入れれば、次の平均値を得られる。

    $$
    \mu_{{k+1}} = a \mu_k + b \Rightarrow E[f(x)] = f(E[x])
    $$


- 分散

$$
\begin{aligned}
P_{k+1} &= E[(f(x_k)- \mu_{k+1})^2] \\
&= E[((ax_k + b) - (a \mu_k + b))^2] \\
&= E[(ax_k - a\mu_k)^2] \\
&= a^2E[(x_k-\mu_k)^2] \\
&= a^2 P_k
\end{aligned}
$$


しかし、例えば次のように状態方程式が非線形の場合

$$
x_{k+1} = f(x_k) = x_k^2 , \quad x_k \sim \mathcal{N}(0,1)
$$

現在の平均値$\mu_k$はゼロとなる。

$$
\mu_k = E[x_k] = 0
$$

線形状態方程式の関係性と同じく、「現状態の平均値を状態方程式へ入れると、次状態の推定値が得られる」と考えると、次のようになる。

$$
f(E[x]) = f(\mu_k) = f(0) = 0
$$

しかし実際に次の状態の平均は次のようになる。

$$
\mu_{k+1} = E[f(x)] = E[x_k^2] = E[(x_k - \mu_k)^2] = 1
$$

よって、非線形方程式では以下が成り立たない。

$$
f(E[x]) = 0 \neq E[f(x)] = 1
$$

これは元の状態が平均$0$の周りに

$$
-1 , \quad 0 , \quad 1
$$

のようにばらついていたとすると、$f(x)=x^2$を通すことによって、以下のように正側へ変換される。

$$
1 , \quad 0 , \quad 1
$$

よって、変換後の平均値は0ではなく、正側へ移動する。

よって、平均値$\mu_k$だけでなく、分布を代表するいくつかの点も$f$を通過させてデータの平均$\mu_{k+1}$と分散$P_{k+1}$を近似する。


## シグマポイント

### 1次元の場合

#### 目的

現在の状態を以下とする。

$$
x \sim \mathcal{N}(\mu, P)
$$

この分布全体を非線形関数$f(x)$に通すことは大変なので、少数の代表点でこの分布を置き換えることを考える。

例えば次の3点とする。この3点を適当に選ぶのではなく、元の状態の平均$\mu$と分散$P$も再現できることが必要である。これをシグマポイントと呼ぶ。

$$
\chi_0 , \chi_1, \chi_2
$$

#### シグマポイントの設定

次のようにシグマポイントを平均とその左右対称に設定する。

$$
\chi_0 = \mu, \quad \chi_1 = \mu + c , \quad \chi_2 = \mu -c
$$

それぞれのポイントに対する重みを以下とする。

$$
W_0, W_1, W_2
$$

平均に対して左右対称であるため、以下のように設定する。

$$
W_1 = W_2 = W
$$

また、この3点で元の分布を表現するため、重みの合計が1である必要がある。

$$
W_0 + 2W = 1
$$

#### シグマポイントのよる平均

シグマポイントから計算した平均を$\hat{\mu}$とすると、

$$
\begin{aligned}
\hat{\mu} &= W_0 \chi_0 + W \chi_1 + W \chi_2 \\
&= W_0 \mu + W (\mu + c) + W (\mu - c) \\
&= (W_0 + 2W)\mu \\
&= \mu
\end{aligned}
$$

したがって、 $\hat{\mu} = \mu$となり、シグマポイントにより、元の分布の平均値を再現できる。

#### シグマポイントによる分散

シグマポイントから計算した分散を$\hat{P}$とすると、

$$
\begin{aligned}
\hat{P} &= \sum_i W_i(\chi_i - \mu)^2 \\
&= W_0 (\mu - \mu)^2 + W(\mu+c - \mu)^2 + W(\mu - c - \mu)^2 \\
&= Wc^2 + Wc^2 \\
&= 2Wc^2
\end{aligned}
$$

これを元の分散と一致させたいので、次のような係数$c$が必要である。

$$
2Wc^2 = P \quad \Rightarrow \quad c = \sqrt{\frac{P}{2W}}
$$

これより、シグマポイントは$\sqrt{P}$に比例した距離となり、このシグマポイントで元の分布の分散を再現することが出来る。

#### シグマポイントの例

仮に$W_0$と以下のように決める。

$$
W_0 = \frac{1}{2}
$$

重みの総和を1とするので、$W$は次のように決まる。

$$
W_0 + 2W = 1 \rightarrow W = \frac{1}{4}
$$

そして、シグマポイントとの係数$c$は次のようになる。

$$
c = \sqrt{\frac{P}{2W}} = \sqrt{2P}
$$

したがって、シグマポイントは次のように決まる。

$$
\chi_0 = \mu, \quad \chi_1 = \mu + \sqrt{2P} , \quad \chi_2 = \mu - \sqrt{2P}
$$


#### シグマポイントを非線形関数へ通す

状態$x$を前節と同じく以下とする。

$$
x \sim \mathcal{N}(0, 1)
$$

これによりシグマポイントは次のようになる。

$$
\chi_0 = 0, \quad \chi_1 = \sqrt{2} , \quad \chi_2 = -\sqrt{2}
$$

これを以下の非線形関数に通過させた後の平均と分散を求める。

$$
f(x) = x^2
$$

変換後のシグマポイントを$Y_i$とすると、

$$
Y_i = f(\chi_i)
$$

より、以下となる。

$$
Y_0 = 0, \quad Y_1 = 2, \quad Y_2 = 2
$$

変換後の平均を求めると"1"となる。

$$
\begin{aligned}
\mu_y &= \sum_i W_i Y_i \\
&= \frac{1}{2}(0) + \frac{1}{4}(2) + \frac{1}{4}(2) \\
&= 1
\end{aligned}
$$

これにより、確率変数である$x$の分布点をいくつも$f$に通さずとも、本当の期待値である、

$$
E[f(x)] = E[x^2] =1
$$

と一致する。

変換後の値から分散を求めると、"1"となっている。

$$
\begin{aligned}
P_y &= \sum_i W_i (Y_i - \mu_y)^2 \\
&= \frac{1}{2}(0 - 1)^2 + \frac{1}{4}(2 - 1)^2 + \frac{1}{4}(2-1)^2 \\
&= \frac{1}{2} + \frac{1}{4} + \frac{1}{4} \\
&= 1
\end{aligned}
$$

これは本当の分散は

$$
\begin{aligned}
\text{Var}(f(x)) &= E[(f(x) - E[f(x)])^2] \\
&= E[(x^2 - 1)^2] \\
&= E[x^4] -2 E[x^2] + 1 \\
\end{aligned}
$$

$E[x^2]=1 , E[x^4]=3$であるため、$\text{Var}(f(x)) = 2$ となり、$\text{Var}(f(x)) \neq P_y$ となっている。

これは、シグマポイントは元の平均と分散を再現していても、任意の非線形変換後の平均と分散を厳密に再現するわけではないことを示している。

## Scaled Unsented Transform

この方法は上節の「1次元の状態に対し、シグマポイントの3点をどう配置するか」を$n$次元へ一般化したものである。そして、上節では考慮できていなかった変換後の分散について考慮する方法でもある。

### シグマポイントの数

状態を$n$次元のベクトル$x$とし、平均ベクトル$\mu$と共分散行列$P$に従う確率変数であると捉える。

$$
x \in \mathbf{R}^{n}, \quad x \sim \mathcal{N}(\mu, P)
$$

シグマポイントは状態の数$n$に応じて、以下のように$2n + 1$となる。

$$
\boxed{
\begin{aligned}
\chi_0 &= \mu \\
\chi_i &= \mu + \sqrt{(n + \lambda)} (\sqrt{P})_i , \quad i,...,n \\
\chi_{n+i} &= \mu - \sqrt{(n + \lambda)}(\sqrt{P})_i, \quad i,...,n \\
\end{aligned}
}
$$

$\boxed{\lambda = \alpha^2(n+\kappa) -n}$ である。

$(\sqrt{P})_i$は$P$の平方行列の$i$番目の列を表す。

これは1次元で確認したときに $chi = \mu \pm c$として分散を再現するために$c \propto \sqrt{P}$ であった。多次元でも同じ関係となる。

共分散行列$P$は正定値対称行列であるため、Cholesky分解などで、$P=LL^T$を求めることが出来る。$L$は下三角行列である。

2次元であれば$P$を次のように分解できるので、

$$
P = LL^T \quad \rightarrow \quad L = \begin{bmatrix} l_{11} & 0 \\ l_{12} & l_{22} \end{bmatrix} = \begin{bmatrix} \mid & \mid \\ l_1 & l_2 \\ \mid & \mid \end{bmatrix} 
$$

シグマポイントは$\mu$を中心に以下のように設定できる。

$$
\begin{aligned}
\mu + \sqrt{n+\lambda} \ l_1 , \quad \mu - \sqrt{n+\lambda} \ l_1 \\
\mu + \sqrt{n+\lambda} \ l_2 , \quad \mu - \sqrt{n+\lambda} \ l_2 \\
\end{aligned}
$$


### 重みの設定

#### 平均用の重み

$$
\boxed{
W_0^{(m)} = \frac{\lambda}{n + \lambda}, \quad W_i^{(m)} = \frac{1}{2(n+\lambda)}, \space i = 1, ..., 2n
}
$$

重みの合計は"1"であることは、以下で確認できる。

$$
\begin{aligned}
\sum_{i=0}^{2n} W_i^{(m)} &= \frac{\lambda}{n + \lambda} + 2n \frac{1}{2(n+\lambda)} \\
&= \frac{\lambda + n}{n + \lambda} \\
&= 1
\end{aligned}
$$

#### 共分散用の重み

$$
\boxed{
W_0^{(c)} = \frac{\lambda}{n+\lambda} + (1 - \alpha^2 + \beta), \quad W_i^{(c)} = \frac{1}{2(n+\lambda)}, \space i=1,...,2n
}
$$

中央のシグマポイントだけ、次のようになっている。

$$
W_0^{(c)} = W_0^{(m)} + (1 - \alpha^2 + \beta)
$$

この補正によって、単に「元の状態の平均と共分散を再現する」だけよりも、非線形変換後の共分散をうまく近似できるようになる。

#### $\alpha, \beta, \kappa$ について

|パラメータ|主な役割|
|:---|:---|
|$\alpha$|シグマポイントを平均からどれくら広げるか|
|$\beta$|分布の形について事前情報を共分散計算に反映|
|$\kappa$|シグマポイントは位置の追加調整|
|$\lambda$|上記から決まるスケーリング係数|

特に

$$
\lambda = \alpha^2(n+\kappa) - n \quad \Rightarrow \quad n+\lambda = \alpha^2(n+\kappa)
$$

となっているため、シグマポイントは以下のように平均$\mu$から$\alpha\sqrt{(n+\kappa)}$に比例した距離になる。よって、$\alpha$を小さくすると、シグマポイントは平均の近くに集まる。

$$
\chi_i = \mu + \sqrt{(n + \lambda)} (\sqrt{P})_i , \quad \chi_{n+i} = \mu - \sqrt{(n + \lambda)}(\sqrt{P})_i
$$

### 状態が1次元の例で確認

下記の状態が1次元の例にScaled Unsented Transform のシグマポイントを適用する。

$$
x \sim \mathcal{N}(0,1), \quad f(x) = x^2
$$

パラメータを以下のようにする。

$$
\alpha=1, \quad \kappa=0, \quad \beta=2 
$$

すると、

$$
\begin{array}{c}
\lambda = \alpha^2(n + \kappa) - n  =1^2(1 + 0) -1 = 0 \\
\sqrt{P} = 1 \\
\sqrt{(n + \lambda)}\sqrt{P} = 1 
\end{array}
$$

シグマポイントは以下となる。

$$
\chi_0 = 0, \quad \chi_1 = 1, \quad \chi_2 = -1
$$

平均用重みは以下となり、

$$
W_0^{(m)} = 0, \quad W_1^{(m)} = \frac{1}{2},  \quad W_2^{(m)} = \frac{1}{2}
$$

共分散用重みは以下となる。

$$
W_0^{(c)} = (1-\alpha^2+\beta) = 2 , \quad W_1^{(c)} = \frac{1}{2},  \quad W_2^{(c)} = \frac{1}{2}
$$

### 元の分布の再現性

- 平均

$$
\begin{aligned}
\hat{\mu} &= W_0^{(m)} \chi_0 + W_1^{(m)} \chi_1 + W_2^{(m)} \chi_2 \\
&= \frac{1}{2}(1) + \frac{1}{2}(-1) \\
&= 0 = \mu
\end{aligned}
$$

- 共分散

$$
\begin{aligned}
\hat{P} &= W_0^{(c)}(\chi_0 - \mu)^2 + W_1^{(c)}(\chi_1 - \mu)^2 + W_2^{(c)}(\chi_2 - \mu)^2 \\
&= \frac{1}{2}(1^2) + \frac{1}{2}(1^2) \\
&= 1 = P
\end{aligned}
$$ 

### 変換後の分布

シグマポイントを$f(x)=x^2$に通すと、以下のポイントになる。

$$
Y_0 = 0 \quad Y_1 = 1, Y_2 = 1
$$

ここから、平均と共分散を求める。

- 平均

    $$
    \begin{aligned}
    \mu_y &= W_0^{(m)} Y_0 + W_1^{(m)} Y_1 + W_2^{(m)} Y_2 \\
    &= \frac{1}{2}(1) + \frac{1}{2}(1) \\
    &= 1
    \end{aligned}
    $$

    これは$E[f(x)]=E[x^2]=1$と一致する。

- 共分散

    $$
    \begin{aligned}
    P_y &= W_0^{(c)}(Y_0 - \mu_y)^2 + W_1^{(c)}(Y_1 - \mu_y)^2 + W_2^{(c)}(Y_2 - \mu_y)^2 \\
    &= 2(0 - 1)^2 + \frac{1}{2}(1- 1)^2 + \frac{1}{2}(1- 1)^2 \\
    &= 2
    \end{aligned}
    $$ 

    これは$\text{Var}(f(x))=\text{Var}(x^2)=2$と一致する。


今回の状態の分布は正規分布としたため、$\beta=2$として、計算を行った。これにより、シグマポイントとその重みを用いて、元の分布の平均と分散、非線形変換後の分布の平均と分散を求めることが出来た。

## UKF

### 状態推定を行う全体の流れ

非線形な状態方程式と観測方程式を次のように定義する。

$$
x_{k+1} = f(x_k, u_k) + w_k , \quad z_k = h(x_k) + v_k
$$

$w_k$はシステムノイズ、$v_k$は観測ノイズであり、それぞれ以下の平均値と共分散行列の正規分布に従うとする。

$$
w_k \sim \mathcal{N}(0, Q_k) , \quad v_k \sim \mathcal{N}(0, R_k)
$$

### 0. 事前の設定

時刻$k$で、推定値へ測定値を反映した事後推定の推定状態$\hat{x}_{k|k}$と誤差共分散行列の結果を持っているとする。

$$
\hat{x}_{k|k}, \quad P_{k|k}
$$

#### 0.1 シグマポイント生成

事後推定結果から予測ステップで用いるシグマポイントを計算する。上付きの括弧付きの数字はシグマポイントの数を表している。

$$
\begin{aligned}
\chi_{k|k}^{(0)} &= \hat{x}_{k|k} \\
\chi_{k|k}^{(i)} &= \hat{x}_{k|k} + \left[ \sqrt{(n + \lambda)} \ \sqrt{P_{k|k}} \right]_i, \quad i=1,...,n \\
\chi_{k|k}^{(n+i)} &= \hat{x}_{k|k} - \left[ \sqrt{(n + \lambda)} \ \sqrt{P_{k|k}} \right]_i, \quad i=1,...,n \\
\end{aligned}
$$

### 1. 予測ステップ

#### 1.1 シグマポイントを状態方程式へ通す

シグマポイントを状態方程式へ通し、次の時刻でどの状態になるのかを予測する。

$$
\chi_{k+1|k}^{(i)} = f(\chi_{k|k}^{(i)}, u_k) , i = 0, ... , 2n
$$

#### 1.2 次状態の予測値を求める

状態方程式を通過したシグマポイントの平均値を次状態の予測値として求める。

$$
\hat{x}_{k+1|k} = \sum_{i=0}^{2n} W_i^{(m)} \chi_{k+1|k}^{(i)}
$$

これは、$E[f(x_k)]$の近似であり、$\hat{x}_{k+1|k} \approx E[x_{k+1}]$ である。

#### 1.3 次状態の共分散を求める

状態方程式を通過したシグマポイントが平均からどれくらいばらついているかを以下で求める。

$$
P_{k+1|k} = \sum_{i=0}^{2n} W_i^{(c)}\left( \chi_{k+1|k}^{(i)} - \hat{x}_{k+1|k} \right) \left( \chi_{k+1|k}^{(i)} - \hat{x}_{k+1|k} \right)^T + Q_k
$$

$Q_k$はシステムノイズの共分散行列である。


ここまでで、状態と誤差共分散行列の事前推定値が求まる。

$$
\hat{x}_{k+1|k} , \quad P_{k+1|k}
$$

### 2. 観測予測

#### 2.1 事前推定値からシグマポイントを作る

$$
\hat{x}_{k+1|k} , \quad P_{k+1|k}
$$

から再びシグマポイント $\psi_{k+1|k}^{(i)}$ を生成する。

$$
\begin{aligned}
\psi_{k+1|k}^{(0)} &= \hat{x}_{k+1|k} \\
\psi_{k+1|k}^{(i)} &= \hat{x}_{k+1|k} + \left[ \sqrt{(n + \lambda)} \ \sqrt{P_{k+1|k}} \right]_i, \quad i=1,...,n \\
\psi_{k+1|k}^{(n+i)} &= \hat{x}_{k+1|k} - \left[ \sqrt{(n + \lambda)} \ \sqrt{P_{k+1|k}} \right]_i, \quad i=1,...,n \\
\end{aligned}
$$

#### 2.2 シグマポイントを観測方程式に通す

$$
Z_{k+1}^{(i)} = h(\psi_{k+1|k}^{(i)})
$$

これは、事前推定のシグマポイントが、センサを通じてどのように観測されるのかを求めている。

状態空間のシグマポイントが、観測空間へ移されている。

#### 2.3 予測観測値を求める

シグマポイントの平均を予測観測値として以下で求める。

$$
\hat{z}_{k+1|k} = \sum_{i=0}^{2n} W_i^{(m)} Z_{k+1}^{(i)}
$$

これはモデルから予想される観測値である。

#### 2.4 観測共分散を求める

予測観測値がどれくらい不確かかを以下の観測共分散により求める。

$$
S_{k+1} = \sum_{i=0}^{2n} W_i^{(c)} \left( Z_{k+1}^{(i)} - \hat{z}_{k+1|k} \right) \left(Z_{k+1}^{(i)} - \hat{z}_{k+1|k} \right)^T + R_{k+1}
$$

$R$はセンサノイズの共分散である。

#### 2.5 状態と観測の関係を求める

##### 交差共分散

予測状態のズレと予測観測値のズレが、どれくらい一致しているのかを以下の交差共分散から求める。

$$
P_{xz} = \sum_{i=0}^{2n} W_i^{(c)} \left( \psi_{k+1}^{(i)} - \hat{x}_{k+1|k} \right) \left(Z_{k+1}^{(i)} - \hat{z}_{k+1|k} \right)^T 
$$

これは例えば位置$x$が大きくなるならセンサ値$z$も大きくなるなら強い相関が出る。これにより、観測値の誤差から状態をどちらへ修正すればよいかの情報になる。

### 3. Kalman Gain

カルマンゲインを以下で求める。

$$
K_{k+1} = P_{xz}S_{k+1}^{-1}
$$

$P_{xz}$で「観測誤差と状態誤差の関係」を見て、$S^{-1}$で「その観測がどのくらい不確かか」を考慮する。

### 4. 測定値による更新

#### 4.1 状態の更新

システムの測定値を

$$
z_{k+1}
$$

とする。ここからinnovationを次で計算する。

$$
\tilde{z}_{k+1} = z_{k+1} - \hat{z}_{k+1|k}
$$

そして、事後推定の状態を、事前推定状態とKalman Gainを使い以下で計算する。

$$
\hat{x}_{k+1|k+1} = \hat{x}_{k+1|k} + K_{k+1} \tilde{z}_{k+1}
$$

これは、「予測状態 + Kalman Gain $\times$ 予測と実測の差」の計算であり、線形Kalman Filterの計算と同じ考え方である。

#### 4.2 共分散の更新

最後に、誤差共分散行列を以下で更新する。

$$
P_{k+1|k+1} = P_{k+1|k} - K_{k+1} S_{k+1} K_{k+1}^T
$$

これにより、このステップでの事後推定値が求まる。

$$
\hat{x}_{k+1|k+1}, \quad P_{k+1|k+1}
$$

次のステップではこの事後推定値を使い、シグマポイントの生成から計算を始める。


## 各ステップの詳細

### 1. 予測ステップ

時刻$k$で事後推定が以下のように得られているとする。

$$
x_k \sim \mathcal{N}(\hat{x}_{k|k}, P_{k|k})
$$

状態方程式は次のように、$x_k,u_k$により次状態を計算する非線形な$f$に、プロセスノイズが加算されたものとして定義されているとする。

$$
x_{k+1} = f(x_k, u_k) + w_k, \quad w_k \sim \mathcal{N}(0,Q_k)
$$

予測ステップでは、実機の観測値$z_{k+1}$が得られていない状態で、次時刻の推定状態と不確かさを示す共分散を計算する。

##### シグマポイント

$\hat{x}_{k|k}, P_{k|k}$から作る下記のシグマポイントは、少ない点数で$\mathcal{N}(\hat{x}_{k|k}, P_{k|k})$の分布を表している。

$$
\chi_{k|k}^{(0)}, \space \chi_{k|k}^{(i)}, \space \chi_{k|k}^{(n+i)}, \space i=1,...,2n
$$

これをプロセスノイズ$w_k$を無視した状態方程式に代入し、不確かさも含んだ次時刻のシグマポイントを計算する。

$$
\chi_{k+1|k}^{(i)} = f(\chi_{k|k}^{(i)}, u_k)
$$

$f$による変換が行われたシグマポイント$\chi_{k+1|k}^{(i)}$には$P_{k|k}$の情報を含んでいる。これを重みを付けて積算したものを次時刻の予測状態として以下で求めている。

$$
\hat{x}_{k+1|k} = \sum_i W_i^{(m)} \chi_{k+1|k}^{(i)}
$$

これは分布の平均を計算していることと同じであり、確率変数である$x_k$を$f$に通過させ、その期待値を計算することを近似している。

$$
\hat{x}_{k+1|k} \approx E[f(x_k, u_k)]
$$


#### 共分散について

非線形な$f$によって変換されたシグマポイント$\chi_{k+1|k}^{(i)}$の共分散を考える。$\chi_{k+1|k}^{(i)}$の平均$\hat{x}_{k+1|k}$からの各シグマポイントのずれを以下のように表す。

$$
\delta \chi_{k+1}^{(i)} = \chi_{k+1|k}^{(i)} - \hat{x}_{k+1|k}
$$

多次元の分散は以下で計算される。

$$
P = E[(x-\mu)(x-\mu)^T]
$$

シグマポイントで分散を近似すると次のようになる。

$$
P_{k+1|k}^f = \sum_{i} W_i^{(c)} \left( \delta \chi_{k+1}^{(i)} \right) \left(\delta \chi_{k+1}^{(i)}\right)^T
$$

この分散$P_{k+1|k}^f $は$w_k$を考慮していない。

つまり以下のように、現在の不確かさ$P_{k|k}$の$f$による非線形変換が行われている。

$$
P_{k|k} \xrightarrow{f} P_{k+1|k}^f 
$$

#### プロセスノイズについて

状態方程式は以下であった。

$$
x_{k+1} = f(x_k, u_k) + w_k, \quad w_k \sim \mathcal{N}(0, Q_k)
$$

次状態の平均は以下となる。

$$
\begin{aligned}
E[x_{k+1}] &= E[f(x_k, u_k) + w_k] \\
&= E[f(x_k, u_k)] + E[w_k] \\
&= E[f(x_k, u_k)]
\end{aligned}
$$

よって、プロセスノイズはこの近似式には追加されない。

$$
\hat{x}_{k+1|k} = \sum_i W_i^{(m)} \chi_{k+1|k}^{(i)} \approx E[f(x_k, u_k)]
$$

一方、共分散は、誤差の分散として次のように計算する。

まず、状態の平均値からのズレは以下のようになる。

$$
x_{k+1} - E[x_{k+1}] = \underbrace{f(x_k, u_k) - E[f(x_k, u_k)]}_{\delta f} + w_k
$$

これを用いて分散は次のように計算される。

$$
\begin{aligned}
P_{k+1|k} &= E[(\delta f + w_k)(\delta f + w_k)^T] \\
&= E[\delta f \delta f^T] + E[\delta f w_k^T] + E[w_k \delta f^T] + E[w_k w_k^T]
\end{aligned}
$$

通常$w_k$は$x_k$と独立だと仮定する。$\delta f$は確率変数$x_k$による関数だと考えると、交差項はゼロとなる。

$$
E[\delta f w_k^T] = E[w_k \delta f^T] = 0
$$

また、$w_k$の分散は以下のように$Q_k$となる。

$$
E[w_k w_k^T] =Q_k
$$

よって、分散は次のようになる。

$$
P_{k+1|k} = E[\delta f \delta f^T] + Q_k
$$

シグマポイント$\chi_{k+1|k}^{(i)}$を用いて$P_{k+1|k}$は次のように近似できる。

$$
P_{k+1|k} = \sum_{i} W_i^{(c)} \left( \chi_{k+1|k}^{(i)} - \hat{x}_{k+1|k} \right) \left( \chi_{k+1|k}^{(i)} - \hat{x}_{k+1|k} \right)^T + Q_k
$$



#### 二つの不確かさ

分散 $P_{k+1|k}$は二つの不確かさがあるといえる。

$$
P_{k+1|k} = P_{k|K} \text{が} f \text{を通って伝播した不確かさ} + Q_k
$$

例えば、ロボットなら、状態として記述される位置や速度でこれは確率変数として推定され、その不確かさとして分散$P_{k|k}$がある。

一方、

- モデル化していない摩擦
- 微小な外乱
- アクチュエータのばらつき
- 離散化誤差
- モデル化誤差

などをプロセスノイズとして表現したものが$Q_k$である。これは、状態遷移する間に新しく追加される不確かさである。

### 2. 観測予測

観測方程式は次のように非線形な$h$により定義されているとする。

$$
z_{k+1} = h(x_{k+1}) + v_{k+1}, \quad v_{k+1} \sim \mathcal{N}(0, R_{k+1})
$$

予測ステップで求める事前推定値はシグマポイントを非線形な$f$を通過させ、かつプロセスノイズが加算されたものとして、以下のように不確かさとともに求まっている。

$$
\hat{x}_{k+1|k}, \quad P_{k+1|k}
$$

この不確かさは正規分布であると仮定しているため、次状態は以下の正規分布に従うと捉えることが出来る。

$$
x_{k+1} \sim \mathcal{N}(\hat{x}_{k+1|k}, P_{k+1|k})
$$

$h$も非線形である単純に平均値の$\hat{x}_{k+1|k}$を通過させるだけの下記の式の左辺と、次状態の観測値の期待値である右辺が、予測ステップと同じく一致しない。

$$
h(E[x_{k+1}]) \neq E[h(x_{k+1})]
$$

よって、事前推定値である$(\hat{x}_{k+1|k},P_{k+1|k})$からシグマポイント$\psi_{k+1|k}^{(i)}$を作り、それらを非線形な観測方程式$h$を通過させる。

これにより、状態空間から観測空間への変換が生じる。

#### 変換後のシグマポイント

観測方程式を通過したシグマポイントを$Z_{k+1}^{(i)}$とする。

$$
Z_{k+1}^{(i)} = h(\psi_{k+1|k}^{(i)})
$$

$Z_{k+1}^{(i)}$は、予測状態の分布を表すシグマポイントを$h$に通過して得られた、観測空間における代表点である。

この$h$を通過したシグマポイントの平均(期待値)を以下の近似計算で求める。

$$
\hat{z}_{k+1|k} = \sum_i W_i^{(m)} Z_{k+1}^{(i)}
$$

これは、次状態の観測値の期待値の近似である。

$$
\hat{z}_{k+1|k} \approx E[h(x_{k+1})]
$$

これは、今の推定状態が正しいと仮定した場合、センサは平均的にどれくらいの値になるかを求めている。仮定としている理由は、$Z_{k+1}$は$k$時刻の事後推定値から観測値によるフィードバックなしに以下のように変換してきているからである。

$$
(\hat{x}_{k|k}, P_{k|k}) \rightarrow \chi_{k+1|k} \xrightarrow{f} (\hat{x}_{k+1|k},P_{k+1|k}) \rightarrow \psi_{k+1|k} \xrightarrow{h} Z_{k+1}
$$

#### 共分散について

非線形な$h$によって変換されたシグマポイント$Z_{k+1}^{(i)}$の共分散を考える。$Z_{k+1}^{(i)}$の平均$\hat{z}_{k+1|k}$からの各ポイントのズレを以下のように表す。

$$
\delta Z_{k+1}^{(i)} = Z_{k+1}^{(i)} - \hat{z}_{k+1|k}
$$

これを用いて共分散は次のように近似できる。

$$
S_{k+1}^h = \sum_i W_i^{(c)} \left( \delta Z_{k+1}^{(i)} \right) \left( \delta Z_{k+1}^{(i)} \right)^T
$$

この分散$S_{k+1}^h$は$v_k$を考慮していない。

これは不確かさ$P_{k+1|k}$の$h$による非線形変換である。

$$
P_{k+1|k} \xrightarrow{h} S_{k+1}^h
$$

#### 観測雑音について

観測方程式は以下であった。

$$
z_{k+1} = h(x_{k+1}) + v_{k+1}, \quad v_{k+1} \sim \mathcal{N}(0, R_{k+1})
$$

測定値の平均は以下となる。

$$
\begin{aligned}
E[z_{k+1}] &= E[h(x_{k+1}) + v_{k+1}] \\
&= E[h(x_{k+1})] + E[v_{k+1}] \\
&= E[h(x_{k+1})]
\end{aligned}
$$

よって、観測雑音は以下の近似式には追加されない。

$$
\hat{z}_{k+1|k} = \sum_i W_i^{(m)} Z_{k+1}^{(i)} \approx E[h(x_{k+1})]
$$

一方、共分散は誤差の分散として次のように計算する。

まず、観測値の平均値からのズレは以下のようになる。

$$
z_{k+1} - E[z_{k+1}] = \underbrace{h(x_{k+1}) - E[h(x_{k+1})]}_{\delta h} + v_{k+1}
$$

これを用いて分散は以下のように計算される。

$$
\begin{aligned}
S_{k+1} &= E[(\delta h + v_{k+1})(\delta h + v_{k+1})^T] \\
&= E[\delta h \delta h^T] + E[\delta h v_{k+1}^T] + E[v_{k+1} \delta h^T] + E[v_{k+1} v_{k+1}^T]
\end{aligned}
$$

通常$v_{k+1}$と$x_{k+1}$は独立と仮定する。$\delta h$は$x_{k+1}$の関数だと考えると、交差項はゼロとなる。

$$
E[\delta h v_{k+1}^T] = E[v_{k+1} \delta h^T] = 0
$$

また、$v_{k+1}$の分散は以下のように$R_{k+1}$となる。

$$
E[ v_{k+1} v_{k+1}^T] = R_{k+1}
$$

よって分散は次のようになる。

$$
\begin{aligned}
S_{k+1} = E[\delta h \delta h^T] + R_{k+1}
\end{aligned}
$$

シグマポイント$Z_{k+1}$を用いて$S_{k+1}$は次のように近似できる。

$$
\begin{aligned}
S_{k+1} = \sum_i W_i^{(c)} \left( Z_{k+1}^{(i)} - \hat{z}_{k+1|k} \right) \left( Z_{k+1}^{(i)} - \hat{z}_{k+1|k} \right)^T + R_{k+1}
\end{aligned}
$$

## 状態空間と観測空間の不確かさ

予測ステップでは非線形関数$f$による状態予測に対して、

$$
x_k \xrightarrow{f} x_{k+1}
$$

次の状態空間の不確かさとなった。

$$
P_{k+1|k} = \underbrace{\sum_i W_i ^{(c)} \delta \chi_i \delta \chi_i^T}_{P_{k|k}\text{が}f\text{を通って伝播した不確かさ}} + \underbrace{Q_k}_{\text{プロセスノイズ}}
$$

観測予測では、非線形関数$h$による観測予測に対して、

$$
x_{k+1} \xrightarrow{h} z_{k+1}
$$

次の観測空間の不確かさとなった。

$$
S_{k+1} = \underbrace{\sum_i W_i ^{(c)} \delta Z_i \delta Z_i^T}_{P_{k+1|k}\text{が}h\text{を通って観測空間へ伝播した不確かさ}} + \underbrace{R_{k+1}}_{\text{観測ノイズ}}
$$

これらは以下の変換の結果である。

$$
\underbrace{P_{k|k}}_{\text{状態空間 時刻}k} \xrightarrow{f} \underbrace{P_{k+1|k}}_{\text{状態空間 時刻}k+1} \xrightarrow{h} \underbrace{S_{k+1}}_{\text{観測空間 時刻}k+1}
$$

状態空間から観測空間への次元数は以下のようになる。

$$
x_{k+1} \in \mathbf{R}^{n_x} \xrightarrow{h} z_{k+1} \in \mathbf{R}^{n_z} \\
$$

これに伴い、共分散行列の次元は以下のようになる。

$$
P_{k+1|k} \in \mathbf{R}^{n_x \times n_x} \xrightarrow{h} S_{k+1} \in \mathbf{R}^{n_z \times n_z} \\
$$


## 3. 交差共分散について

これまで求めたものは、以下のものであった。

- $P_{k+1|k}$ : 状態空間における状態自身のばらつき
- $S_{k+1}$ : 観測空間における観測値自身のばらつき

これにもう一つの考え方として、交差共分散があり、ここからカルマンゲインが求まる。

### 観測値のズレに応じて、状態をどのように修正するか

システムの実測値が以下で得られたとする。

$$
z_{k+1}
$$

ここから、予測観測値との差(innovation)が次のように計算される。

$$
\tilde{z}_{k+1} = z_{k+1} - \hat{z}_{k+1|k}
$$

ここで例えば、観測モデルが単純な以下の形をしていたとする。

$$
z = x
$$

$\tilde{z}_{k+1} > 0$ であるならば、予測観測値よりも実測値のほうが大きいということであるため、予測状態$x$を大きくすると、予測観測値は実測値に近づく。

また、観測モデルが以下の負の比例関係である場合、$\tilde{z}_{k+1} > 0$ であるならば、予測観測値よりも実測値のほうが大きいということであるため、予測状態$x$を小さくすると、予測観測値は大きくなり実測値に近づく。

$$
z = -x
$$

このように、「状態のズレと観測のズレがどのように連動しているのか」を表す量が交差共分散$P_{xz}$である。この関係を利用することで、innovationから推定状態$x$をどのように修正すべきかを決めることが出来る。

### 共分散について

#### 分散について

確率変数$x$があり、その平均は期待値を用いて以下で計算できる。

$$
\mu_x = E[x]
$$

平均との差は $x-\mu$であり、これは、確率変数$x$が平均からどれだけずれたかを示す。

これを2乗して平均すると、分散が計算できる。

$$
\text{Var}(x) = E[(x - \mu)^2]
$$

2乗するため、$x>\mu$でも$x<\mu$でも分散は正になる。分散は「$x$自身が平均からどれくらいばらつくか」を表す指標である。

#### 二つの確率変数 $x,z$の分散

$x,z$のそれぞれの平均を以下と置く。

$$
\mu_x = E[x] , \quad \mu_z = E[z]
$$

平均との差は以下となる。

$$
\delta x = x - \mu_x, \quad \delta z = z - \mu_z
$$

これらの積 「$\delta x \delta z$」を考える。

#### $\delta x \delta z$の符号

例えば

$$
\delta x >0 , \quad \delta z > 0
$$

なら、

$$
\delta x \delta z > 0
$$

であり、$x$が平均より大きいとき、$z$も平均より大きいとなる。

逆に、

$$
\delta x < 0 , \quad \delta z < 0
$$

でも、

$$
\delta x \delta z > 0
$$

であり、$x$が平均より小さいとき、$z$も平均より小さいとなる。

この二つの関係はどちらも「$x$と$z$が同じ方向へ動いた」結果といえる。

一方、

$$
\delta x > 0 , \quad \delta z < 0 \quad \rightarrow \quad \delta x \delta z < 0
$$

この逆でも

$$
\delta x < 0 , \quad \delta z > 0 \quad \rightarrow \quad \delta x \delta z < 0
$$

となる。これは「$x$と$z$が互いに逆方向へ動いた」結果といえる。

### 平均化する

1回だけ 「$\delta x \delta z$」を求めても、たまたまであった可能性がある。そこで、確率分布全体について期待値を取る。

$$
\text{Cov}(x,z) = E[(x - \mu_x) (z - \mu_z)]
$$

これが共分散 Covariance である。

つまり、Covarianceは、「$x$と$z$の平均との差が互いにどのように動くのか」を平均的に評価していることになる。

### 簡単な例

#### 例1. $z=2x$

$E[x]=0$なら、$z=2x$より、$E[z]=0$となる。これより共分散は以下となる。

$$
\begin{aligned}
\text{Cov}(x,z) &= E[(x-0)(z-0)] \\
&= 2E[x^2]
\end{aligned}
$$

$E[x^2] = \text{Var}(x)$ であるため、

$$
\text{Cov}(x,z) = 2 \text{Var}(x) > 0
$$

となる。$\text{Cov}(x,z)>0$であるため、

$$
x > 0 \Rightarrow z > 0, \quad x < 0 \Rightarrow z <0
$$

という正の比例の関係を示す。

#### 例2. $z=-2x$

この場合、共分散は

$$
\begin{aligned}
\text{Cov}(x,z) &= E[(x-0)(z-0)] \\
&= -2E[x^2] \\
&= -2 \text{Var}(x)
\end{aligned}
$$

$\text{Cov}(x,z)<0$であるため、

$$
x > 0 \Rightarrow z < 0, \quad x < 0 \Rightarrow z > 0
$$

という負の比例関係を示す。



### UKFで計算したもの

状態については、確率変数$x$の平均からのズレを$\delta x$と定義し、

$$
\delta x = x_{k+1} - E[x_{k+1}]
$$

以下のシグマポイントを用いた分散計算により、「状態と状態の共分散」を求めた。

$$
\begin{aligned}
P_{k+1|k} &= \sum_{i} W_i^{(c)} \left( \chi_{k+1|k}^{(i)} - \hat{x}_{k+1|k} \right) \left( \chi_{k+1|k}^{(i)} - \hat{x}_{k+1|k} \right)^T + Q_k \\
&\approx E[\delta x \delta x^T]
\end{aligned}
$$

観測値については、確率変数$z$の平均からのズレを$\delta z$と定義し、

$$
\delta z = z_{k+1} - E[z_{k+1}]
$$

以下のシグマポイントを用いた分散計算により、「観測値と観測値の共分散」を求めた。

$$
\begin{aligned}
S_{k+1} &= \sum_i W_i^{(c)} \left( Z_{k+1}^{(i)} - \hat{z}_{k+1|k} \right) \left( Z_{k+1}^{(i)} - \hat{z}_{k+1|k} \right)^T + R_{k+1} \\
&\approx E[\delta z \delta z^T]
\end{aligned}
$$

ここから求めたいものは、以下の$\delta x$と$\delta z$の共分散$P_{xz}$であり、これは「状態と観測値の交差共分散」となる。

$$
P_{xz} = E[\delta x \delta z^T]
$$

#### 状態をベクトル、観測値をスカラーとした場合

例えば状態$x$を

$$
x = \begin{bmatrix} q \\ \dot{q} \end{bmatrix}
$$

観測値を$z$という1次元センサ値とする。

状態$x$も観測値$z$もある分布に従う確率変数である。するとそれらの平均値からのズレは以下となる。

$$
\delta x = \begin{bmatrix} \delta q \\ \delta \dot{q} \end{bmatrix} = \begin{bmatrix}  q - \mu_q \\ \dot{q} - \mu_{\dot{q}} \end{bmatrix} , \quad \delta z = z - \mu_z
$$

これより、交差項は以下となる。

$$
\delta x \delta z^T = \begin{bmatrix} \delta q \\ \delta \dot{q} \end{bmatrix} \delta z = \begin{bmatrix} \delta q \delta z \\ \delta \dot{q} \delta z \end{bmatrix}
$$

交差項の期待値を取ると、次のようになる。

$$
P_{xz} = \begin{bmatrix}
\text{Cov}(q,z)\\
\text{Cov}(\dot{q}, z)
\end{bmatrix} = 
\begin{bmatrix}
E[(q - \mu_q) (z - \mu_z)] \\
E[(\dot{q} - \mu_{\dot{q}}) (z - \mu_z)]
\end{bmatrix}
$$

共分散は多数の実現値の全体的な統計量である。

$$
(\delta q_1, \delta z_1), (\delta q_2, \delta z_2), ... , (\delta q_i, \delta z_i)
$$

$$
\text{Cov}(q,z) > 0
$$

なら、$\delta z$ が正となる実現に対して $\delta q$ も正となる傾向がある。または、$\delta z$ が負となる実現に対して $\delta q$ も負となる傾向がある。

一方、

$$
\text{Cov}(\dot{q},z) < 0
$$

なら、$\delta z$ が正となる実現に対して $\delta \dot{q}$ は負となる傾向がある。または、$\delta z$ が負となる実現に対して $\delta \dot{q}$ は正となる傾向がある。

これらは統計量であり、実際には以下の状態・観測モデルから計算を行う。そして、このモデルは実際のシステムの挙動を近似した数式である。<br>

$$
\begin{aligned}
x_{k+1} &= f(x_k, u_k) + w_k, \quad w_k \sim \mathcal{N}(0, Q_k) \\
z_{k+1} &= h(x_{k+1}) + v_{k+1}, \quad v_{k+1} \sim \mathcal{N}(0, R_{k+1})
\end{aligned}
$$

$z_{k+1}$は状態$x_{k+1}$を非線形な観測方程式$h$を介して得られるため、この変換により、$x_{k+1}$と$z_{k+1}$が正比例の関係であるなら共分散は正となり、負の比例関係であれば共分散は負となるといった形である。比例関係は線形であるが、実際には$h$は非線形であるため、簡単には計算できない。そこで、シグマポイントによる共分散の近似計算を行う。

$$
x_{k+1} \xrightarrow{h} z_{k+1}
$$


### $P_{xz}$の計算

交差期待値の$P_{xz}$は次のように求まる。

$$
P_{xz} = E[\delta x \delta z^T]
$$

予測ステップで以下の事前推定値を得た。

$$
x_{k+1} \sim \mathcal{N}(\hat{x}_{k+1|k}, P_{k+1|k})
$$

そして、この事前推定値からシグマポイント$\psi_{k+1|k}^{(i)}$を作り、観測方程式へ通過させて$Z_{k+1}^{(i)}$を求めた。

$$
Z_{k+1}^{(i)} = h\left(\psi_{k+1|k}^{(i)} \right)
$$

この関係に、確率変数の平均とのズレを対応させると以下のようになる。

$$
\begin{aligned}
\delta x &= x_{k+1} - \hat{x}_{k+1|k} \\
\delta z &= \underbrace{h(x_{k+1}) - E[h(x_{k+1})]}_{\delta h} + w_k = \delta h + v_k
\end{aligned}
$$

これを$P_{xz}$に代入して展開すると、

$$
\begin{aligned}
P_{xz} &= E[\delta x (\delta h + v_{k+1})^T] \\
&= E[\delta x \delta h^T] + E[\delta x v_{k+1}^T]
\end{aligned}
$$

$\delta x, \delta h$は$x_{k+1}$に依存した値であり、観測ノイズ$v_{k+1}$は$x_{k+1}$と独立であると仮定すれば、

$$
E[\delta x v_{k+1}^T] = 0
$$

したがって、

$$
P_{xz} = E[\delta x \delta h^T]
$$

この共分散計算をシグマポイントを使って近似すると以下である。

$$
P_{xz} = \sum_{i=0}^{2n} W_i^{(c)}\left(\psi_{k+1|k}^{(i)} - \hat{x}_{k+1|k} \right) \left(Z_{k+1}^{(i)} - \hat{z}_{k+1|k} \right)^T
$$


## 4. Kalman Gain

これまでに、以下の共分散を求めた。

$$
P = E[\delta x \delta x^T], \quad P_{xz} = E[\delta x \delta z^T], \quad S = E[\delta z \delta z^T]
$$

ここで、$\delta x, \delta z$は以下であり、$\delta x$は確率変数である$x_{k+1}$とその平均$\hat{x}_{k+1|k}$から計算した値で、真値であり確率変数である。$\delta z$について同じく真値であり確率変数である。

$$
\delta x = x_{k+1} - \hat{x}_{k+1|k}, \quad \delta z = z_{k+1} - \hat{z}_{k+1|k}
$$

このとき、観測値のズレである$\delta z$から状態のズレ$\delta x$を推定したいとする。

これを次の線形な推定則で行うと仮定する。

$$
\widehat{\delta x} = K \delta z
$$

本当の状態のズレは$\delta x$で、推定したズレは$K \delta z$であるため、状態の推定誤差$e$は次となる。

$$
e = \delta x - K \delta z
$$

この$e$も確率変数であるため、この誤差の大きさは分散を用いて、以下で計算できる。

$$
P_e = E[e \ e^T]
$$

この$P_e$をできるだけ小さくする$K$を求める。この$K$がKalman Gainとなる。

$P_e$を展開して、共分散を代入する。$P_{xz}^T = E[\delta z \delta x^T] = P_{zx}$を用いている。

$$
\begin{aligned}
P_e &= E[e \ e^T] = E[(\delta x - K \delta z)(\delta x - K \delta z)^T] \\
&= E[\delta x \delta x^T] -  E[\delta x \delta z^T] \ K^T- K E[\delta z \delta x^T] + K E[\delta z \delta z^T] K^T \\
&= P - P_{xz} K^T - K P_{zx} + K S K^T
\end{aligned}
$$

$P_e$は行列であり、これをここから最小化するために、行列のトレースを適用する。

--------------------------------------------
例えば誤差$e$を2次元として

$$
e = \begin{bmatrix} e_1 \\ e_2 \end{bmatrix}
$$

とする。

共分散の計算では$E[e e^T]$を計算するので、次のような行列になる。

$$
e \ e^t = \begin{bmatrix}
e_1^2 & e_1 e_2 \\
e_2 e_1 & e_2^2
\end{bmatrix}
$$

最小化を行うためのスカラー値として評価するため、対角項の$e_1^2,e_2^2$を用いるとすると、行列のトレースにより、

$$
\text{tr}(ee^T) = e_1^2 + e_2^2
$$

このように２乗和の形で評価することが出来る。これにより、最小点を見つけることが出来る。

------------------------------------------------------

$P_e$にトレースを行ったものを評価関数$J(K)$とする。$J(K)$は$K$により変化する値であることを示している。

$$
\begin{aligned}
J(K) &= \text{tr}(P_e) \\
&= \text{tr}(P) - \text{tr}(P_{xz} K^T )- \text{tr}(K P_{zx}) +\text{tr}(K S K^T)
\end{aligned}
$$